# R6 · Protección de datos y ética

**Responsable:** Dan (Rol B)  
**Revisor:** Leo (Rol D)

## Qué queda resuelto

- [x] Argumento con las leyes 19.628 y 21.719
- [x] Identificar celdas pequeñas y aplicar la supresión (`src.proteccion.supresion`)
- [x] Advertencias de interpretación (testeo, comparación entre países)
- [x] Excelente: condados de EE.UU. e información perdida

In [ ]:
import sys, pathlib
RAIZ = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(RAIZ))
import duckdb
import pandas as pd
from src.proteccion.supresion import suprimir, informacion_perdida

DW_PATH = RAIZ / 'data' / 'dw_covid.duckdb'

## Argumento con las leyes 19.628 y 21.719

Los conteos agregados por territorio no identifican a personas naturales (no son datos personales ni sensibles de salud), pero en poblaciones pequeñas con conteos de 1 a 4 existe riesgo de reidentificación indirecta (ver `docs/proteccion_datos.md`).

## Identificar celdas pequeñas y aplicar la supresión (`src.proteccion.supresion`)

In [ ]:
con = duckdb.connect(str(DW_PATH), read_only=True)
df_us = con.execute('''
    SELECT t.combined_key, t.population AS poblacion, f.casos_nuevos, f.fallecidos_nuevos
    FROM fact_covid f
    JOIN dim_territorio t ON f.territorio_key = t.territorio_key
    WHERE t.country_region = 'US' AND t.nivel_granularidad = 'condado' AND t.population > 0
''').df()
con.close()

df_sup = suprimir(df_us.head(20), ['casos_nuevos', 'fallecidos_nuevos'], umbral_poblacion=5000)
display(df_sup.head(10))

## Advertencias de interpretación (testeo, comparación entre países)

Documentadas en `docs/proteccion_datos.md` y visibles en la barra lateral de `dashboard/app.py`.

## Excelente: condados de EE.UU. e información perdida

In [ ]:
perdida_us = informacion_perdida(df_us, ['casos_nuevos', 'fallecidos_nuevos'], umbral_poblacion=5000)
display(perdida_us)